# HomeWork14

## Импорты, seed и среда

In [1]:
# Импорт библиотек
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from sklearn.metrics import pairwise_distances
from sentence_transformers import SentenceTransformer
import faiss
import kagglehub

# Фиксируем Seed
SEED = 47
random.seed(SEED)
np.random.seed(SEED)

# torch
USE_TORCH = True
if USE_TORCH:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## База знаний и первичный анализ

In [2]:
# Скачиваем датасет SQuAD (параграфы из Wikipedia)
path = kagglehub.dataset_download("stanfordu/stanford-question-answering-dataset")
squad_file = os.path.join(path, "train-v1.1.json")

with open(squad_file, "r", encoding="utf-8") as f:
    squad_data = json.load(f)

# Извлечение всех параграфов
documents = []
for article in squad_data["data"]:
    for paragraph in article["paragraphs"]:
        documents.append({
            "id": len(documents)+1,
            "text": paragraph["context"]
        })

doc_df = pd.DataFrame(documents)
print("Количество исходных документов:", len(doc_df))

# Покажем примеры
print("Примеры документов:")
display(doc_df.sample(5, random_state=SEED))

Количество исходных документов: 18896
Примеры документов:


,id,text
1003,1004,Portuguese universities have existed since 129...
8232,8233,The first record of the name Israel (as ysrỉꜣr...
14933,14934,"Before Europeans arrived, the Philadelphia are..."
8113,8114,When maintenance is performed on asphalt pavem...
124,125,"Montana's motto, Oro y Plata, Spanish for ""Gol..."


**Предметная область:** SQuAD (Stanford Question Answering Dataset) – параграфы из статей Wikipedia, содержащие фактологическую информацию. Это идеальный полигон для retrieval/RAG, так как:
- тексты чётко структурированы;
- легко формулировать запросы с ожидаемыми источниками;
- подходит для оценки качества поиска.

## Чанкинг документов

In [3]:
chunk_size = 256
overlap = 64

def chunk_text(text, chunk_size=chunk_size, overlap=overlap):
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += (chunk_size - overlap)
    return chunks

all_chunks = []
for _, row in doc_df.iterrows():
    chunks = chunk_text(row["text"])
    for i, c in enumerate(chunks):
        all_chunks.append({
            "doc_id": row["id"],
            "chunk_id": i+1,
            "text": c
        })

chunks_df = pd.DataFrame(all_chunks)
print("Общее число чанков:", len(chunks_df))
print("Примеры чанков:")
display(chunks_df.sample(5, random_state=SEED))

Общее число чанков: 81481
Примеры чанков:


,doc_id,chunk_id,text
74724,17432,2,"ring the 8th century, trading and frequently a..."
30761,7903,1,The Gospel of John and particularly the first ...
57798,13730,4,"parison, family resemblance). Catholic theolog..."
30690,7887,2,rain that had been meant to feed the poor for ...
75850,17678,4,"uational awareness)"" (JP 1-02)."


## Эмбеддинги и индекс FAISS

In [4]:
model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)
embedder.to(device)

texts = chunks_df["text"].tolist()
embeddings = embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True)
print("Форма эмбеддингов:", embeddings.shape)

embedding_dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(embedding_dim)
faiss_index.add(embeddings)
print("Количество векторов в индексе:", faiss_index.ntotal)

# Пример поиска
top_k = 3
example_queries = [
    "What is the capital of USA?",
    "Who was the MVP of Super Bowl XXXIV",
    "Which author created Sherlock Holmes?"
]

print("\nПримеры поиска:")
for q in example_queries:
    q_emb = embedder.encode([q], convert_to_numpy=True)
    D, I = faiss_index.search(q_emb, top_k)
    print(f"\nЗапрос: {q}")
    for rank, idx in enumerate(I[0], start=1):
        print(f"Ранг {rank}: {chunks_df.iloc[idx]['text'][:150]}...")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1482.60it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 2547/2547 [11:08<00:00,  3.81it/s]


Форма эмбеддингов: (81481, 384)
Количество векторов в индексе: 81481

Примеры поиска:

Запрос: What is the capital of USA?
Ранг 1: The capital city, Washington, District of Columbia, is a federal district located on land donated by the state of Maryland. (Virginia had also donated...
Ранг 2: ted onshore of the Atlantic Ocean, Boston is the only state capital in the contiguous United States with an oceanic coastline....
Ранг 3: Boston (pronounced i/ˈbɒstən/) is the capital and largest city of the Commonwealth of Massachusetts in the United States. Boston also served as the hi...

Запрос: Who was the MVP of Super Bowl XXXIV
Ранг 1: The year 2000 brought heightened interest in the AFL. Then-St. Louis Rams quarterback Kurt Warner, who was MVP of Super Bowl XXXIV, was first noticed ...
Ранг 2: In 2012, Madonna performed at Super Bowl XLVI halftime show, visualized by Cirque Du Soleil and Jamie King and featured special guests LMFAO, Nicki Mi...
Ранг 3: w Jersey, which hosted Super Bowl XLVI

## Контрольные запросы и оценка retrieval

In [5]:
control_queries = [
    {"query": "Capital of USA", "keywords": ["capital", "United States"]},
    {"query": "Super Bowl XXXIV MVP", "keywords": ["Kurt Warner", "MVP"]},
    {"query": "Sherlock Holmes author", "keywords": ["Conan Doyle", "author"]},
    {"query": "Beer alcohol content", "keywords": ["alcohol", "volume", "beer"]},
    {"query": "Fastest insect runners", "keywords": ["Cockroaches", "fastest", "insect"]},
    {"query": "CAA ATC responsibilities", "keywords": ["CAA", "ATC", "airports"]},
    {"query": "British ales strength", "keywords": ["British", "ales", "4%"]},
    {"query": "Singapore surrender 1942", "keywords": ["Singapore", "Japanese", "1942"]},
    {"query": "Nutritional status studies", "keywords": ["nutritional", "status", "experiments"]},
    {"query": "London Great Plague", "keywords": ["London", "Plague", "1665"]}
]

def is_relevant(text, keywords):
    text_lower = text.lower()
    return any(kw.lower() in text_lower for kw in keywords)

k_values = [1, 3, 5]
metrics = {k: {"hit": 0, "recall_sum": 0} for k in k_values}
eval_results = []

for q_data in control_queries:
    q_emb = embedder.encode([q_data["query"]], convert_to_numpy=True)
    D, I = faiss_index.search(q_emb, max(k_values))

    retrieved_texts = [chunks_df.iloc[idx]['text'] for idx in I[0]]
    retrieved_ids = [chunks_df.iloc[idx]['doc_id'] for idx in I[0]]

    for k in k_values:
        top_k_texts = retrieved_texts[:k]
        relevant_count = sum(1 for t in top_k_texts if is_relevant(t, q_data["keywords"]))
        if relevant_count > 0:
            metrics[k]["hit"] += 1
        metrics[k]["recall_sum"] += min(relevant_count, 1)

    rank_first = -1
    hit_at_k = {k: 0 for k in k_values}
    for k in k_values:
        for i in range(k):
            if is_relevant(retrieved_texts[i], q_data["keywords"]):
                hit_at_k[k] = 1
                if rank_first == -1:
                    rank_first = i + 1
                break

    eval_results.append({
        "query": q_data["query"],
        "expected_source": ", ".join(q_data["keywords"]),
        "retrieved_sources": ";".join(map(str, retrieved_ids[:5])),
        "hit_at_k": hit_at_k[5],
        "rank_of_first_relevant": rank_first
    })

print(f"{'k':<5} | {'Hit@k':<10} | {'Recall@k':<10}")
print("-" * 30)
for k in k_values:
    hit_rate = metrics[k]["hit"] / len(control_queries)
    recall_rate = metrics[k]["recall_sum"] / len(control_queries)
    print(f"{k:<5} | {hit_rate:<10.2f} | {recall_rate:<10.2f}")

# Сохранение в артефакты
os.makedirs("artifacts", exist_ok=True)
pd.DataFrame(eval_results).to_csv("artifacts/retrieval_eval.csv", index=False)

k     | Hit@k      | Recall@k  
------------------------------
1     | 1.00       | 1.00      
3     | 1.00       | 1.00      
5     | 1.00       | 1.00      


## Небольшой эксперимент с параметрами retrieval

In [6]:
# ID документов, которые содержат ответы на первые 5 запросов (из retrieval_eval.csv)
relevant_doc_ids = [16129, 3592, 10111, 6135, 14077]
# Берём эти документы + добавляем немного шума (первые 100 документов)
sample_docs = doc_df[doc_df['id'].isin(relevant_doc_ids) | (doc_df.index < 100)]

chunk_sizes = [200, 400]
test_queries = control_queries[:5]  # только первые 5 запросов

results = []

for size in chunk_sizes:
    # Чанкинг выбранных документов
    chunks = []
    for _, row in sample_docs.iterrows():
        start = 0
        text = row['text']
        while start < len(text):
            end = min(start + size, len(text))
            chunk = text[start:end].strip()
            if chunk:
                chunks.append(chunk)
            start += (size - 50)

    # Эмбеддинги и индекс
    emb = embedder.encode(chunks, convert_to_numpy=True, show_progress_bar=False)
    idx = faiss.IndexFlatL2(emb.shape[1])
    idx.add(emb)

    hits = 0
    for q_data in test_queries:
        q_emb = embedder.encode([q_data["query"]], convert_to_numpy=True)
        D, I = idx.search(q_emb, 1)
        retrieved_text = chunks[I[0][0]].lower()
        if any(kw.lower() in retrieved_text for kw in q_data["keywords"]):
            hits += 1

    hit_rate = hits / len(test_queries)
    results.append({"chunk_size": size, "Hit@1": hit_rate})
    print(f"Chunk size: {size}, Hit@1: {hit_rate:.2f}")

pd.DataFrame(results)

Chunk size: 200, Hit@1: 1.00
Chunk size: 400, Hit@1: 0.80


,chunk_size,Hit@1
0,200,1.0
1,400,0.8


## Обновление базы знаний и переиндексация

In [7]:
# Новые документы
new_docs = [
    {"id": 99901, "text": "Qt 6.5 introduced new features to Qt Quick. Integration with C++ remained stable."},
    {"id": 99902, "text": "FAISS is a library for efficient vector similarity search and clustering."},
    {"id": 99903, "text": "Sentence Transformers provide dense vector representations for sentences."}
]
new_df = pd.DataFrame(new_docs)

# Чанкинг новых документов
new_chunks = []
for _, row in new_df.iterrows():
    chunks = chunk_text(row['text'])
    for i, c in enumerate(chunks):
        new_chunks.append({"doc_id": row['id'], "chunk_id": i+1, "text": c})
new_chunks_df = pd.DataFrame(new_chunks)

# Обновление общего DataFrame
chunks_df_updated = pd.concat([chunks_df, new_chunks_df], ignore_index=True)

# Запросы для проверки обновления
update_queries = [
    "What's new in Qt 6.5?",
    "FAISS library purpose",
    "Sentence Transformers usage"
]

# Поиск ДО добавления
before_results = {}
for q in update_queries:
    q_emb = embedder.encode([q], convert_to_numpy=True)
    D, I = faiss_index.search(q_emb, 3)
    before_ids = [chunks_df.iloc[idx]['doc_id'] for idx in I[0]]
    before_results[q] = before_ids

# Добавляем новые векторы в индекс
new_embeddings = embedder.encode(new_chunks_df['text'].tolist(), convert_to_numpy=True)
faiss_index.add(new_embeddings)
print(f"Индекс после добавления: {faiss_index.ntotal} векторов")

# Поиск ПОСЛЕ
update_comparison = []
for q in update_queries:
    q_emb = embedder.encode([q], convert_to_numpy=True)
    D, I = faiss_index.search(q_emb, 3)
    after_ids = [chunks_df_updated.iloc[idx]['doc_id'] for idx in I[0]]

    before_ids = before_results[q]
    changed = "Yes" if before_ids != after_ids else "No"

    update_comparison.append({
        "query": q,
        "before_retrieved_sources": ";".join(map(str, before_ids)),
        "after_retrieved_sources": ";".join(map(str, after_ids)),
        "changed": changed
    })

pd.DataFrame(update_comparison).to_csv("artifacts/retrieval_before_after_update.csv", index=False)
print("Сравнение сохранено в artifacts/retrieval_before_after_update.csv")

Индекс после добавления: 81484 векторов
Сравнение сохранено в artifacts/retrieval_before_after_update.csv


## Mini-RAG

In [8]:
def run_mini_rag(query, k=3):
    q_emb = embedder.encode([query], convert_to_numpy=True)
    D, I = faiss_index.search(q_emb, k)

    contexts = []
    sources = []
    for idx in I[0]:
        row = chunks_df_updated.iloc[idx]
        contexts.append(row['text'])
        sources.append(row['doc_id'])

    # Простой способ формирования ответа: взять первый чанк
    answer = contexts[0] if contexts else "Извините, ответ не найден."

    return {
        "query": query,
        "answer": answer,
        "sources": sources,
        "context": contexts
    }

# Тест
result = run_mini_rag("What library is used for vector search?")
print(f"Запрос: {result['query']}")
print(f"Ответ: {result['answer']}")
print(f"Источники: {result['sources']}")

Запрос: What library is used for vector search?
Ответ: FAISS is a library for efficient vector similarity search and clustering.
Источники: [np.int64(99902), np.int64(5385), np.int64(10584)]


In [9]:
# Создаём примеры для артефактов
test_queries = [
    "What library is used for vector search?",
    "When was Qt 6.5 released?",
    "Who wrote Harry Potter?",
    "What's the weather like in Moscow today?",
    "How do I install FAISS on Windows?"
]

rag_results = []
for q in test_queries:
    res = run_mini_rag(q, k=3)
    rag_results.append({
        "question": q,
        "answer": res['answer'],
        "retrieved_sources": ";".join(map(str, res['sources']))
    })

pd.DataFrame(rag_results).to_csv("artifacts/rag_examples.csv", index=False)
print("Примеры сохранены в artifacts/rag_examples.csv")

Примеры сохранены в artifacts/rag_examples.csv


## Краткий анализ ошибок

1. **Who wrote Harry Potter?** – система выдала нерелевантный фрагмент о "brothers, was the head writer". Причина: в выбранной подвыборке SQuAD отсутствует информация о Дж. К. Роулинг. Векторный поиск нашёл семантически похожий текст (структура "author → ..."), но фактически неверный. Это демонстрирует необходимость гибридного поиска или reranking.

2. **What's the weather like in Moscow today?** – ответ основан на общих описаниях климата, так как в статической базе нет данных реального времени. Система не может отличить запросы, требующие внешних источников. Для таких случаев нужен классификатор out-of-scope запросов.

3. **How do I install FAISS on Windows?** – найден фрагмент про Windows 95, но без инструкции по установке. Причина: база знаний не содержит туториалов, а чанкинг разбил бы инструкцию, если бы она была. Проблему можно решить добавлением технической документации и использованием адаптивного чанкинга.

4. **When was Qt 6.5 released?** – версия найдена, но дата отсутствует, так как в добавленном документе она не была указана. Это ограничение полноты базы знаний.

**Вывод:** качество retrieval определяет успех mini-RAG. Даже при идеальном поиске, если база не содержит ответа, система даст неверный ответ. Для улучшения нужны более богатая база, гибридный поиск и детекция запросов вне компетенции.